# Notebook 72R1: V8.8 Registered-Replay Operator (Colab-safe)

This notebook fixes the previous combined notebook. It does **not** invoke `argparse`, `main()`, or `SystemExit`, so Colab will not try to parse `colab_kernel_launcher.py` arguments.

## Scope

This is a **registered-replay correction runner**. It requires:

- one observed page;
- the corresponding prepared M3 mask;
- either a SAFE_REMOVE mask or a decision NPZ;
- optionally a SAFE_PROTECT mask for historical replay.

It does not infer SAFE_REMOVE for new annotated-only pages. If those decision artefacts do not already exist, this notebook stops with a clear message rather than pretending to run V8.8.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless numpy


## 1. Configuration

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# REQUIRED: edit these four values.
OBSERVED_PATH = ROOT / 'aligned' / 'Page002.png'
M3_PATH = ROOT / 'YOUR_V88_RUN' / 'pages' / 'Page002' / 'm3.png'

# Use exactly ONE of these two decision sources.
SAFE_REMOVE_PATH = ROOT / 'YOUR_V88_RUN' / 'pages' / 'Page002' / 'safe_remove.png'
DECISION_NPZ_PATH = None

# Optional historical registered replay only.
SAFE_PROTECT_PATH = None
ENABLE_SAFE_PROTECT_REPLAY = False

OUTPUT_DIR = ROOT / 'notebook72r1_output' / 'Page002'

METHOD = 'telea'
RADIUS = 3
CROP_MARGIN = 48
PAPER_SIGMA = 45.0
REMOVE_CODE = 1
PROTECT_CODE = 2


## 2. Operator implementation

In [ ]:
import cv2, numpy as np, hashlib, json, platform, sys
from datetime import datetime, timezone

SUPPORTED_IMAGES={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition,message):
    if not bool(condition):
        raise RuntimeError(message)

def sha256_file(path):
    path=Path(path); h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256()
    h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

def read_gray(path):
    path=Path(path)
    require(path.is_file(),f'Missing image: {path}')
    require(path.suffix.lower() in SUPPORTED_IMAGES,f'Unsupported image type: {path}')
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {path}')
    return im

def read_mask(path,shape,label):
    im=read_gray(path)
    require(im.shape==shape,f'{label} shape {im.shape} != observed {shape}')
    return im>0

def padded_components(mask,margin=48,minimum_area=8):
    count,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    H,W=mask.shape; boxes=[]
    for c in range(1,count):
        x,y,w,h,area=stats[c]
        if area>=minimum_area:
            boxes.append((max(0,x-margin),max(0,y-margin),min(W,x+w+margin),min(H,y+h+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                overlap=not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0)
                if overlap:
                    ax0,ay0=min(ax0,bx0),min(ay0,by0); ax1,ay1=max(ax1,bx1),max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def background_surface(observed,authority,sigma=45.0):
    visible=(~authority)&(observed>=200)
    num=cv2.GaussianBlur(observed.astype(np.float32)*visible,(0,0),sigma)
    den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    surface=np.where(den>0.02,num/np.maximum(den,1e-4),fallback)
    return np.clip(surface,0,255).astype(np.uint8)

def reconstruct_residual(observed,residual,method='telea',radius=3,crop_margin=48):
    result=observed.copy(); mask_u8=residual.astype(np.uint8)*255
    flag=cv2.INPAINT_TELEA if method=='telea' else cv2.INPAINT_NS
    for x0,y0,x1,y1 in padded_components(residual,margin=crop_margin):
        crop=observed[y0:y1,x0:x1]; cm=mask_u8[y0:y1,x0:x1]
        candidate=cv2.inpaint(crop,cm,radius,flag)
        local=result[y0:y1,x0:x1]; local[cm>0]=candidate[cm>0]; result[y0:y1,x0:x1]=local
    require(np.array_equal(result[~residual],observed[~residual]),'Residual reconstruction changed a non-residual coordinate')
    return result

def _npz_array(archive,names):
    for name in names:
        if name in archive.files:
            return np.asarray(archive[name]),name
    return None,None

def load_decisions(shape,safe_remove_path=None,safe_protect_path=None,decision_npz_path=None,remove_code=1,protect_code=2):
    provenance={}
    if decision_npz_path is not None:
        path=Path(decision_npz_path); require(path.is_file(),f'Missing decision NPZ: {path}')
        with np.load(path,allow_pickle=False) as z:
            decision,key=_npz_array(z,['m5_decision_map','decision_map'])
            if decision is not None:
                require(decision.ndim==2,'Decision map must be two-dimensional')
                require(decision.shape==shape,f'Decision map shape {decision.shape} != observed {shape}')
                safe_remove=decision==remove_code; safe_protect=decision==protect_code
                provenance.update({'decision_key':key,'decision_values':np.unique(decision).tolist()})
            else:
                sr,sr_key=_npz_array(z,['SAFE_REMOVE','safe_remove'])
                sp,sp_key=_npz_array(z,['SAFE_PROTECT','safe_protect'])
                require(sr is not None,f'No recognised decision array in {path}; keys={z.files}')
                require(sr.shape==shape,f'{sr_key} shape mismatch: {sr.shape} != {shape}')
                safe_remove=sr>0
                if sp is None: safe_protect=np.zeros(shape,bool)
                else:
                    require(sp.shape==shape,f'{sp_key} shape mismatch: {sp.shape} != {shape}')
                    safe_protect=sp>0
                provenance.update({'safe_remove_key':sr_key,'safe_protect_key':sp_key})
        provenance.update({'decision_npz':str(path),'decision_npz_sha256':sha256_file(path)})
        return safe_remove,safe_protect,provenance

    require(safe_remove_path is not None,'Set SAFE_REMOVE_PATH or DECISION_NPZ_PATH in the configuration cell.')
    sr=Path(safe_remove_path); safe_remove=read_mask(sr,shape,'SAFE_REMOVE')
    if safe_protect_path is None: safe_protect=np.zeros(shape,bool)
    else: safe_protect=read_mask(Path(safe_protect_path),shape,'SAFE_PROTECT')
    provenance={'safe_remove_path':str(sr),'safe_remove_sha256':sha256_file(sr),'safe_protect_path':str(safe_protect_path) if safe_protect_path else None}
    return safe_remove,safe_protect,provenance

def write_image(path,image):
    require(cv2.imwrite(str(path),image),f'Could not write image: {path}')

def run_v88_registered_replay(*,observed_path,m3_path,output_dir,safe_remove_path=None,decision_npz_path=None,safe_protect_path=None,enable_safe_protect_replay=False,method='telea',radius=3,crop_margin=48,paper_sigma=45.0,remove_code=1,protect_code=2):
    observed_path=Path(observed_path); m3_path=Path(m3_path); output_dir=Path(output_dir)
    require(observed_path.is_file(),f'Missing observed page: {observed_path}')
    require(m3_path.is_file(),f'Missing M3 mask: {m3_path}')
    if output_dir.exists():
        require(not any(output_dir.iterdir()),f'Output directory must be empty: {output_dir}')
    output_dir.mkdir(parents=True,exist_ok=True)
    observed=read_gray(observed_path); m3=read_mask(m3_path,observed.shape,'M3')
    safe_remove,safe_protect,provenance=load_decisions(observed.shape,safe_remove_path,safe_protect_path,decision_npz_path,remove_code,protect_code)
    require(not np.any(safe_remove&~m3),'SAFE_REMOVE is not a subset of M3')
    require(not np.any(safe_protect&~m3),'SAFE_PROTECT is not a subset of M3')
    require(not np.any(safe_remove&safe_protect),'SAFE_REMOVE and SAFE_PROTECT overlap')
    if safe_protect.any() and not enable_safe_protect_replay:
        raise RuntimeError('SAFE_PROTECT is present. Set ENABLE_SAFE_PROTECT_REPLAY=True only for historical registered replay.')
    m5=m3&~safe_remove&~safe_protect
    residual=reconstruct_residual(observed,m5,method,radius,crop_margin)
    paper=background_surface(observed,m3,paper_sigma)
    corrected=residual.copy(); corrected[safe_remove]=paper[safe_remove]; corrected[safe_protect]=observed[safe_protect]
    outside=int(np.sum(corrected[~m3]!=observed[~m3])); protect=int(np.sum(corrected[safe_protect]!=observed[safe_protect]))
    require(outside==0,f'Outside-M3 immutability failure: {outside} pixels')
    require(protect==0,f'SAFE_PROTECT copy failure: {protect} pixels')
    for name,image in [('observed.png',observed),('m3.png',m3.astype(np.uint8)*255),('safe_remove.png',safe_remove.astype(np.uint8)*255),('safe_protect.png',safe_protect.astype(np.uint8)*255),('m5.png',m5.astype(np.uint8)*255),('residual_reconstruction.png',residual),('paper_estimate.png',paper),('corrected.png',corrected)]: write_image(output_dir/name,image)
    diff=np.zeros_like(observed); diff[corrected!=observed]=255; write_image(output_dir/'difference_from_observed.png',diff)
    manifest={'schema':'music-annotation-v8.8-registered-replay-colab-v2','created_utc':datetime.now(timezone.utc).isoformat(),'observed':str(observed_path),'m3':str(m3_path),'decision_provenance':provenance,'method':method,'radius':radius,'counts':{'m3':int(m3.sum()),'safe_remove':int(safe_remove.sum()),'safe_protect':int(safe_protect.sum()),'m5':int(m5.sum()),'changed':int((corrected!=observed).sum())},'integrity':{'outside_m3_drift':outside,'safe_protect_copy_drift':protect},'hashes':{'observed':array_hash(observed),'M3':array_hash(m3.astype(np.uint8)),'SAFE_REMOVE':array_hash(safe_remove.astype(np.uint8)),'SAFE_PROTECT':array_hash(safe_protect.astype(np.uint8)),'M5':array_hash(m5.astype(np.uint8)),'corrected':array_hash(corrected)},'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'platform':platform.platform()}}
    (output_dir/'run_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True))
    return {'output_dir':output_dir,'corrected':corrected,'m3':m3,'safe_remove':safe_remove,'safe_protect':safe_protect,'m5':m5,'manifest':manifest}

print('Operator loaded. No command-line parser was invoked.')


## 3. Validate configuration before running

In [ ]:
require(Path(OBSERVED_PATH).is_file(),f'OBSERVED_PATH does not exist: {OBSERVED_PATH}')
require(Path(M3_PATH).is_file(),f'M3_PATH does not exist: {M3_PATH}')
require((SAFE_REMOVE_PATH is None)!=(DECISION_NPZ_PATH is None),'Set exactly one of SAFE_REMOVE_PATH or DECISION_NPZ_PATH.')
if SAFE_REMOVE_PATH is not None: require(Path(SAFE_REMOVE_PATH).is_file(),f'SAFE_REMOVE_PATH does not exist: {SAFE_REMOVE_PATH}')
if DECISION_NPZ_PATH is not None: require(Path(DECISION_NPZ_PATH).is_file(),f'DECISION_NPZ_PATH does not exist: {DECISION_NPZ_PATH}')
if SAFE_PROTECT_PATH is not None: require(Path(SAFE_PROTECT_PATH).is_file(),f'SAFE_PROTECT_PATH does not exist: {SAFE_PROTECT_PATH}')
print('Configuration validation: PASS')


## 4. Run

In [ ]:
result=run_v88_registered_replay(
    observed_path=OBSERVED_PATH,
    m3_path=M3_PATH,
    safe_remove_path=SAFE_REMOVE_PATH,
    decision_npz_path=DECISION_NPZ_PATH,
    safe_protect_path=SAFE_PROTECT_PATH,
    enable_safe_protect_replay=ENABLE_SAFE_PROTECT_REPLAY,
    output_dir=OUTPUT_DIR,
    method=METHOD,
    radius=RADIUS,
    crop_margin=CROP_MARGIN,
    paper_sigma=PAPER_SIGMA,
    remove_code=REMOVE_CODE,
    protect_code=PROTECT_CODE,
)
print('Completed:',result['output_dir'])
print(json.dumps(result['manifest']['counts'],indent=2))


## 5. Preview

In [ ]:
import matplotlib.pyplot as plt
obs=cv2.imread(str(OUTPUT_DIR/'observed.png'),0)
cor=cv2.imread(str(OUTPUT_DIR/'corrected.png'),0)
diff=cv2.imread(str(OUTPUT_DIR/'difference_from_observed.png'),0)
fig,ax=plt.subplots(1,3,figsize=(18,8))
for a,im,title in zip(ax,[obs,cor,diff],['Observed','Corrected','Changed pixels']):
    a.imshow(im,cmap='gray',vmin=0,vmax=255); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()
